# Two-way frequency tables

**STA 501 1.0 Descriptive Statistics** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta501.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Two categorical variables](#theory-01)
- [Two numerical variables](#theory-02)
- [Transformations and interpretation](#theory-03)
- [Ordinal and mixed-scale relationships](#theory-04)
- [Association and causation](#theory-05)
- [Cross-sectional and time-series data](#theory-06)
- [Research proposal and final report](#theory-07)
- [Common misleading comparisons](#theory-08)

## Detailed theory · Association, measurement scales and reporting

<a id="theory-01"></a>

### Two categorical variables
A cross-tabulation shows joint counts and margins. Row percentages answer a question conditional on the row category; column percentages condition on the column. Overall percentages use the full sample denominator. Association is visible when conditional distributions differ, rather than merely when cell counts differ.

<a id="theory-02"></a>

### Two numerical variables
A scatterplot reveals direction, form, strength, clusters and unusual points. Sample covariance is
$$s_{xy}=\frac{\sum(x_i-\bar x)(y_i-\bar y)}{n-1},\quad r=\frac{s_{xy}}{s_xs_y}.$$
Correlation is unit-free and bounded by ±1 when both SDs are nonzero. It summarizes linear association; a strong curved relationship can have r near zero.

<a id="theory-03"></a>

### Transformations and interpretation
For nonzero a,c, $\operatorname{Corr}(aX+b,cY+d)=\operatorname{sign}(ac)\operatorname{Corr}(X,Y)$. Translation and positive rescaling preserve correlation; reversing one scale reverses its sign. Changing units does not solve an outlier or nonlinearity problem.

<a id="theory-04"></a>

### Ordinal and mixed-scale relationships
Spearman correlation is Pearson correlation of ranks with average ranks for ties. The shortcut $1-6\sum d_i^2/[n(n^2-1)]$ assumes no ties. For a numerical outcome by nominal group, compare group distributions or appropriate model estimates rather than assigning arbitrary category numbers and calculating Pearson r.

<a id="theory-05"></a>

### Association and causation
Confounding, reverse direction, selection and measurement can produce association. A strong r is not a causal effect, and an r near zero rules out neither nonlinear dependence nor subgroup effects. Examine design and the substantive mechanism.

<a id="theory-06"></a>

### Cross-sectional and time-series data
Cross-sectional cases concern different units over a period; a time series preserves repeated temporal ordering. Trend, seasonality, cycles and irregular variation can coexist. Correlating two trending series can be misleading, and time dependence invalidates independence-based SEs. Label dates and sampling intervals before plotting or smoothing.

<a id="theory-07"></a>

### Research proposal and final report
A proposal states background, question, objectives and planned methods before results exist. A final report includes introduction, collection and cleaning methods, analysis tied to each objective, results, limitations, conclusions and justified recommendations. Separate findings from speculation; include units, denominators, uncertainty and sample restrictions.

<a id="theory-08"></a>

### Common misleading comparisons
Changing an axis origin, comparing counts from different population sizes or using relative growth from a tiny baseline can distort a story. Percentage change is $100(new-old)/old$ when the baseline is nonzero; a percentage-point change subtracts two percentages. Neither is interchangeable with a rate adjusted for different exposure.

## Concepts covered

**Course points:** Two-way frequency tables; row/column/overall percentages; scatterplots; covariance and Pearson correlation; association vs causation.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · Two-way table: use the correct denominator

### Question and assumptions

Women:20 Yes and 30 No; men:30 Yes and 70 No. Compare within-group Yes percentages.

### Procedure

1. Show counts and row/column totals.
2. For a within-group percentage, divide each count by its own row total.
3. Compare the conditional percentages rather than the raw Yes counts.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Group | Yes | No | Row total | Yes % within group |
| --- | --- | --- | --- | --- |
| Women | 20 | 30 | 50 | 40 |
| Men | 30 | 70 | 100 | 30 |

| Denominator choice | Women Yes calculation |
| --- | --- |
| Within women | 20 / 50 = 40% |
| Of all Yes | 20 / 50 = 40% |
| Of all people | 20 / 150 = 13.33% |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


counts=[["Women",20,30],["Men",30,70]]
show_table(["Group","Yes","No","Row total","Yes % within group"],[[g,y,n,y+n,100*y/(y+n)] for g,y,n in counts])
show_table(["Denominator choice","Women Yes calculation"],[["Within women","20 / 50 = 40%"],["Of all Yes","20 / 50 = 40%"],["Of all people","20 / 150 = 13.33%"]])
assert sum(y+n for g,y,n in counts)==150


### Decision and interpretation

Women have the higher observed Yes rate (40% versus 30%) even though men have more Yes responses. In this example the first two percentages happen to agree because their denominators are both 50; that is not generally true.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
g=rng.choice(['F','M'],300); smoke=rng.choice(['yes','no'],300,p=[.28,.72])
tab=np.array([[np.sum((g==a)&(smoke==b)) for b in ['yes','no']] for a in ['F','M']])
print('counts',tab,'row %',np.round(100*tab/tab.sum(1,keepdims=True),1))
x=rng.uniform(-3,3,250); y=2*x+rng.normal(0,2,250); curve=x*x+rng.normal(0,1.5,250)
print('Pearson r linear/curved:',np.corrcoef(x,y)[0,1].round(3),np.corrcoef(x,curve)[0,1].round(3))
fig,ax=plt.subplots(1,2); ax[0].scatter(x,y); ax[1].scatter(x,curve); plt.show()

### Practice set · Association arcade
1. If r=−0.82 between temperature and heating use, explain the sign and strength without claiming causation.
2. Give a curved relationship with Pearson r near zero.
3. A table has row totals 40 and 60; within-row “yes” counts are 10 and 30. Compute row percentages and compare groups.
4. Add one plausible confounder to a study of screen time and sleep duration.

<details><summary>Answer sketches</summary>
1. Strong negative linear association in the observed sample; it does not prove temperature alone caused lower heating use. 2. A symmetric U-shaped relation such as y=x² with x balanced around zero. 3. 25% and 50%; the second row has the higher observed rate. 4. Work schedule, age, stress, or health can affect both screen use and sleep.
</details>

### Debrief
Describe one association and a plausible confounder.

Record the formula, assumptions, and one follow-up question.

## Lab reflection

Which assumptions came from the study design? What changed when you adjusted a parameter?